In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample2") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

# Устанавливаем таймауты и keep-alive как числа (без 's')
# Значения в секундах или миллисекундах (зависит от версии, обычно keepalivetime в сек)
hadoop_conf.set("fs.s3a.threads.keepalivetime", "60") 
hadoop_conf.set("fs.s3a.connection.timeout", "60000")
hadoop_conf.set("fs.s3a.attempts.maximum", "10")
hadoop_conf.set("fs.s3a.connection.establish.timeout", "5000")
hadoop_conf.set("fs.s3a.readahead.range", "65536")

hadoop_conf.set("fs.s3a.multipart.purge.age", "86400")

hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")


In [ ]:
s3_path_yandex = "s3a://dev/avpalatov/yandex/"

# Чтение parquet-файла
df = spark.read.parquet(s3_path_yandex)
df.show(5) 

In [ ]:
renamed_df = (df
                .withColumnRenamed("ym:s:date", "date")
                .withColumnRenamed("ym:s:regionCountry", "country")
                .withColumnRenamed("ym:s:regionCity", "city")
                .withColumnRenamed("ym:s:browserLanguage", "language")
                .withColumnRenamed("ym:s:browser", "browser")
                .withColumnRenamed("ym:s:deviceCategory", "device")
                .withColumnRenamed("ym:s:operatingSystem", "os")
                .withColumnRenamed("ym:s:hour", "hour")
                .withColumnRenamed("ym:s:visits", "visits")
                .withColumnRenamed("ym:s:users", "users")
                .withColumnRenamed("ym:s:newUsers", "new_users")
                .withColumnRenamed("ym:s:pageviews", "page_views")
)

renamed_df.show()

final_df = renamed_df 


In [ ]:
# ⬇️ Параметры подключения к CLICKHOUSE
jdbc_url = 'jdbc:clickhouse://clickhouse01:8123/avpalatov'
db_user = os.getenv('CLICKHOUSE_USER')
db_password = os.getenv('CLICKHOUSE_PASSWORD')
table_name = 'yandex_metrics'

final_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", table_name) \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("truncate", "true") \
    .mode("append") \
    .save()

print("Таблица сохранена в Clickhouse")

In [ ]:
spark.stop()